# FlyWire v783 comparison

This notebook reruns the 17 feeding experiments from `02_v630_reproduction.ipynb` on the v783 release of the FlyWire connectome and compares the left MN9 calls between the two releases. Neuron IDs that changed between releases are matched with fafbseg before anything is run.

A note on the outputs: the simulation cells show "Skipping experiment" because Run all was clicked again after the batch had finished. The runner skips any experiment whose output file already exists, so nothing was simulated twice. The counts and tables further down are read from the saved output files, which are in `results/raw_v783/` in this repository along with the run record (`manifest.json`).

In [37]:
# ATTENTION: only run this cell when on google colab
!git clone https://github.com/philshiu/Drosophila_brain_model.git
!pip install brian2
%cd Drosophila_brain_model

Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 183 (delta 75), reused 67 (delta 67), pack-reused 95 (from 1)
Receiving objects: 100% (183/183), 185.05 MiB | 33.15 MiB/s, done.
Resolving deltas: 100% (83/83), done.
Updating files: 100% (19/19), done.
/content/Drosophila_brain_model/Drosophila_brain_model/Drosophila_brain_model


In [38]:
from model import run_exp
from model import default_params as params
import utils as utl
from brian2 import Hz

config = {
    'path_res'  : './results/example',                              # directory to store results
    'path_comp' : './2023_03_23_completeness_630_final.csv',        # csv of the complete list of Flywire neurons
    'path_con'  : './2023_03_23_connectivity_630_final.parquet',    # connectivity data
    'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
}

Point the model at v783 and save results to a new folder.

In [39]:
config['path_comp'] = './Completeness_783.csv'
config['path_con']  = './Connectivity_783.parquet'

from google.colab import drive
drive.mount('/content/drive')
import os
config['path_res'] = '/content/drive/MyDrive/fly_project/results_v783/batch1'
os.makedirs(config['path_res'], exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Check the inputs exist in v783 before running anything. You should see 2 neurons for each, nothing missing, and True.

In [40]:
import pickle
import pandas as pd

with open('sez_neurons.pickle', 'rb') as f:
    sez = pickle.load(f)
lookup = {k.casefold(): k for k in sez}

ids_783 = set(pd.read_csv(config['path_comp']).iloc[:, 0].astype('int64'))
MN9_LEFT = 720575940660219265
first = ['roundup', 'FMIn']

for name in first:
    ids = sez[lookup[name.casefold()]]
    missing = [i for i in ids if i not in ids_783]
    print(f"{name}: {len(ids)} input neurons, missing in v783: {missing}")
print("Left MN9 present in v783:", MN9_LEFT in ids_783)

roundup: 2 input neurons, missing in v783: []
FMIn: 2 input neurons, missing in v783: []
Left MN9 present in v783: True


Run both experiments and write a run record. This is the start of the run record (manifest.json). It records the exact data files (by hash), the software versions, the settings and the inputs used. It's saved after each experiment, so a disconnect doesn't lose it.

In [41]:
import json, hashlib, sys, subprocess, datetime, brian2

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

params['r_poi'] = 50 * Hz
params['n_run'] = 30

manifest = {
    'release': 'v783', 'batch': 'batch1',
    'seed_status': 'unseeded (run_exp has no seed argument)',
    'code_commit': subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                                  capture_output=True, text=True).stdout.strip(),
    'python': sys.version.split()[0], 'brian2': brian2.__version__,
    'path_comp': config['path_comp'], 'sha256_comp': sha256(config['path_comp']),
    'path_con': config['path_con'], 'sha256_con': sha256(config['path_con']),
    'r_poi_hz': 50, 'n_run': 30, 't_run_s': float(params['t_run']),
    'readout_left': MN9_LEFT, 'experiments': {},
}
now = lambda: datetime.datetime.now().isoformat(timespec='seconds')

for name in first:
    key = lookup[name.casefold()]
    started = now()
    run_exp(exp_name=f'{name}_50Hz', neu_exc=sez[key], params=params, **config)
    manifest['experiments'][name] = {
        'input_ids': [int(i) for i in sez[key]], 'started': started, 'finished': now()}
    with open(f"{config['path_res']}/manifest.json", 'w') as f:
        json.dump(manifest, f, indent=1)

>>> Skipping experiment roundup_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/roundup_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment FMIn_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/FMIn_50Hz.parquet exists and force_overwrite = False


Read the left MN9 results. What to look for:

- **roundup** should come out near the v630 figures (2,363 spikes, with spikes in all 30 trials). Close enough means the pipeline works. Way off means stop and check before going further.
- **FMIn** is the case to watch. If it spikes in some trials, the call stays positive, and the zero reported by suifei's flywire-fly-lab was most likely its 3-trial batch missing it. If it has no spikes in all 30 trials, that's a flip, and the protocol calls for a confirmation pair. Either way, record the trial and spike counts, not just the yes or no call.

In [42]:
df = utl.load_exps([f"{config['path_res']}/{n}_50Hz.parquet" for n in first])
print("Trials present:", df.groupby('exp_name')['trial'].nunique().to_dict())

baseline = {'roundup': (2363, 30), 'FMIn': (18, 10)}   # v630: spikes, trials with a spike
for name in first:
    d = df[(df['exp_name'] == f'{name}_50Hz') & (df['flywire_id'] == MN9_LEFT)]
    print(f"{name}: v783 {len(d)} spikes in {d['trial'].nunique()} of 30 trials "
          f"({len(d)/30:.2f} Hz) | v630 {baseline[name][0]} spikes in {baseline[name][1]} trials")

Trials present: {'FMIn_50Hz': 30, 'roundup_50Hz': 30}
roundup: v783 2314 spikes in 30 of 30 trials (77.13 Hz) | v630 2363 spikes in 30 trials
FMIn: v783 8 spikes in 8 of 30 trials (0.27 Hz) | v630 18 spikes in 10 trials


Install fafbseg and log in to FlyWire. The token comes from Colab Secrets (under the name `FLYWIRE_TOKEN`), so it is never written into the notebook.

In [43]:
!pip install -q fafbseg
from google.colab import userdata
from fafbseg import flywire

flywire.set_default_dataset('public')
flywire.set_chunkedgraph_secret(userdata.get('FLYWIRE_TOKEN'), overwrite=True)

Default dataset set to "public".
Token succesfully stored.


Look up the v783 IDs for the TH_VUM input and the right MN9 readout.

In [44]:
th_vum_old = [int(i) for i in sez[lookup['th_vum']]]
mn9_right_old = [720575940645521262]

res = flywire.update_ids(th_vum_old + mn9_right_old, timestamp='mat_783')
res['role'] = ['TH_VUM input'] * len(th_vum_old) + ['right MN9 readout']
res['in_model_v783_file'] = res['new_id'].isin(ids_783)
res

,old_id,new_id,confidence,changed,role,in_model_v783_file
0,720575940616857174,720575940610482883,0.997919,True,TH_VUM input,True
1,720575940645521262,720575940618238523,0.999673,True,right MN9 readout,True


Do the cell-type labels agree between releases?

In [45]:
old_ids = th_vum_old + mn9_right_old
new_ids = res['new_id'].tolist()

cols = ['root_id', 'super_class', 'cell_class', 'cell_type', 'hemibrain_type', 'side']
a630 = flywire.search_annotations(old_ids, materialization=630)
a783 = flywire.search_annotations(new_ids, materialization=783)
display(a630[[c for c in cols if c in a630.columns]])
display(a783[[c for c in cols if c in a783.columns]])

,root_id,super_class,cell_class,cell_type,hemibrain_type,side
0,720575940645521262,motor,brain_motor_neuron,CB0701,NaN,left
1,720575940616857174,central,NaN,CB3958,NaN,center


,root_id,super_class,cell_class,cell_type,hemibrain_type,side
0,720575940618238523,motor,brain_motor_neuron,CB0701,NaN,left
1,720575940610482883,central,NaN,CB3958,NaN,center


Is each neuron wired about the same way in both of the model's data files? This tests what actually matters for the simulation: how many connections the neuron has, and whether its output is excitatory or inhibitory.

In [46]:
import numpy as np
comp630 = pd.read_csv('2023_03_23_completeness_630_final.csv', index_col=0)
con630  = pd.read_parquet('2023_03_23_connectivity_630_final.parquet')
comp783 = pd.read_csv(config['path_comp'], index_col=0)
con783  = pd.read_parquet(config['path_con'])

def wiring(root, comp, con):
    i = comp.index.get_loc(root)
    w_out = con.loc[con['Presynaptic_Index'] == i, 'Excitatory x Connectivity']
    w_in  = con.loc[con['Postsynaptic_Index'] == i, 'Excitatory x Connectivity']
    return {'out_partners': len(w_out), 'out_synapses': int(w_out.abs().sum()),
            'out_sign': int(np.sign(w_out.sum())), 'in_partners': len(w_in),
            'in_synapses': int(w_in.abs().sum())}

rows = []
for role, old, new in zip(res['role'], res['old_id'], res['new_id']):
    rows.append({'role': role, 'release': 'v630', 'id': old, **wiring(old, comp630, con630)})
    rows.append({'role': role, 'release': 'v783', 'id': new, **wiring(new, comp783, con783)})
wiring_table = pd.DataFrame(rows)
wiring_table

,role,release,id,out_partners,out_synapses,out_sign,in_partners,in_synapses
0,TH_VUM input,v630,720575940616857174,479,1911,1,335,1453
1,TH_VUM input,v783,720575940610482883,490,2001,1,345,1514
2,right MN9 readout,v630,720575940645521262,75,116,1,242,5874
3,right MN9 readout,v783,720575940618238523,76,117,1,241,5886


Save the match evidence.

In [47]:
ct630 = dict(zip(a630['root_id'], a630['cell_type']))
ct783 = dict(zip(a783['root_id'], a783['cell_type']))
side783 = dict(zip(a783['root_id'], a783['side']))
w = wiring_table.pivot(index='role', columns='release', values='out_synapses')
s = wiring_table.pivot(index='role', columns='release', values='out_sign')

evidence = res[['role', 'old_id', 'new_id', 'confidence']].copy()
evidence['method'] = "fafbseg.flywire.update_ids(timestamp='mat_783'), public dataset"
evidence['cell_type_v630'] = evidence['old_id'].map(ct630)
evidence['cell_type_v783'] = evidence['new_id'].map(ct783)
evidence['flywire_side_label_v783'] = evidence['new_id'].map(side783)
evidence['out_synapses_v630'] = evidence['role'].map(w['v630'])
evidence['out_synapses_v783'] = evidence['role'].map(w['v783'])
evidence['out_sign_same'] = evidence['role'].map(s['v630'] == s['v783'])
evidence['annotation_release'] = 'flyconnectome/flywire_annotations a83b277 (2026-09-29)'
evidence['status'] = 'matched'
evidence.to_csv('/content/drive/MyDrive/fly_project/id_matching_TH_VUM_MN9R.csv', index=False)
evidence

,role,old_id,new_id,confidence,method,cell_type_v630,cell_type_v783,flywire_side_label_v783,out_synapses_v630,out_synapses_v783,out_sign_same,annotation_release,status
0,TH_VUM input,720575940616857174,720575940610482883,0.997919,fafbseg.flywire.update_ids(timestamp='mat_783'...,CB3958,CB3958,center,1911,2001,True,flyconnectome/flywire_annotations a83b277 (202...,matched
1,right MN9 readout,720575940645521262,720575940618238523,0.999673,fafbseg.flywire.update_ids(timestamp='mat_783'...,CB0701,CB0701,left,116,117,True,flyconnectome/flywire_annotations a83b277 (202...,matched


Run TH_VUM on v783 with its matched input.

In [48]:
import json
TH_VUM_783 = [int(res.loc[res['role'] == 'TH_VUM input', 'new_id'].iloc[0])]
MN9_RIGHT_783 = int(res.loc[res['role'] == 'right MN9 readout', 'new_id'].iloc[0])

params['r_poi'] = 50 * Hz
params['n_run'] = 30
started = now()
run_exp(exp_name='TH_VUM_50Hz', neu_exc=TH_VUM_783, params=params, **config)

mpath = f"{config['path_res']}/manifest.json"
with open(mpath) as f:
    manifest = json.load(f)
manifest['readout_right'] = MN9_RIGHT_783
manifest['experiments']['TH_VUM'] = {
    'input_ids': TH_VUM_783, 'input_ids_v630': th_vum_old,
    'matched_by': 'id_matching_TH_VUM_MN9R.csv', 'started': started, 'finished': now()}
with open(mpath, 'w') as f:
    json.dump(manifest, f, indent=1)

>>> Skipping experiment TH_VUM_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/TH_VUM_50Hz.parquet exists and force_overwrite = False


Read both MN9s.

In [49]:
d = utl.load_exps([f"{config['path_res']}/TH_VUM_50Hz.parquet"])
print("Trials present:", d['trial'].nunique())
for side, rid, base in [('left', MN9_LEFT, '0 spikes'), ('right', MN9_RIGHT_783, '78 spikes in 28 trials')]:
    x = d[d['flywire_id'] == rid]
    print(f"{side} MN9: v783 {len(x)} spikes in {x['trial'].nunique()} of 30 trials "
          f"({len(x)/30:.2f} Hz) | v630 {base}")

Trials present: 30
left MN9: v783 0 spikes in 0 of 30 trials (0.00 Hz) | v630 0 spikes
right MN9: v783 101 spikes in 28 of 30 trials (3.37 Hz) | v630 78 spikes in 28 trials


Match the 4 partly changed cell types.

Diatom, kitty, marge and peacock each have one neuron whose ID changed. This cell does the same lookup and checks for all four at once. It adds one more check: whether a "new" ID turns out to be a neuron already in the set, meaning two neurons were merged into one, so it mustn't be stimulated twice.

In [50]:
partial = ['diatom', 'kitty', 'marge', 'peacock']
changed = [(n, int(i)) for n in partial for i in sez[lookup[n.casefold()]] if int(i) not in ids_783]
print(changed)

r = flywire.update_ids([i for _, i in changed], timestamp='mat_783')
r['cell_type_name'] = [n for n, _ in changed]
r['in_model_v783_file'] = r['new_id'].isin(ids_783)
r['already_in_set'] = [nid in {int(i) for i in sez[lookup[n.casefold()]]}
                       for n, nid in zip(r['cell_type_name'], r['new_id'])]

b630 = flywire.search_annotations(r['old_id'].tolist(), materialization=630)
b783 = flywire.search_annotations(r['new_id'].tolist(), materialization=783)
r['cell_type_v630'] = r['old_id'].map(dict(zip(b630['root_id'], b630['cell_type'])))
r['cell_type_v783'] = r['new_id'].map(dict(zip(b783['root_id'], b783['cell_type'])))

for rel, comp, con, idcol in [('v630', comp630, con630, 'old_id'), ('v783', comp783, con783, 'new_id')]:
    ok = r['in_model_v783_file'] if rel == 'v783' else [True] * len(r)
    ww = [wiring(x, comp, con) if k else {} for x, k in zip(r[idcol], ok)]
    r[f'out_synapses_{rel}'] = [x.get('out_synapses') for x in ww]
    r[f'out_sign_{rel}'] = [x.get('out_sign') for x in ww]
r

[('diatom', 720575940614897458), ('kitty', 720575940610056772), ('marge', 720575940610690372), ('peacock', 720575940631237203)]


,old_id,new_id,confidence,changed,cell_type_name,in_model_v783_file,already_in_set,cell_type_v630,cell_type_v783,out_synapses_v630,out_sign_v630,out_synapses_v783,out_sign_v783
0,720575940614897458,720575940633968045,0.991266,True,diatom,True,False,CB4148,CB4148,266,1,266,1
1,720575940610056772,720575940626798736,0.959430,True,kitty,True,False,CB0370,CB0370,1621,-1,1659,-1
2,720575940610690372,720575940633941293,0.999185,True,marge,True,False,CB0812,CB0812,1965,-1,1962,-1
3,720575940631237203,720575940627081534,0.991292,True,peacock,True,False,CB0746,CB0746,845,1,871,1


Save all the match evidence in one file.

In [51]:
ev2 = r[['old_id', 'new_id', 'confidence', 'cell_type_v630', 'cell_type_v783',
         'out_synapses_v630', 'out_synapses_v783']].copy()
ev2.insert(0, 'role', r['cell_type_name'] + ' input')
ev2['out_sign_same'] = r['out_sign_v630'] == r['out_sign_v783']
ev2['method'] = evidence['method'].iloc[0]
ev2['annotation_release'] = evidence['annotation_release'].iloc[0]
ev2['status'] = 'matched'

id_matching = pd.concat([evidence, ev2], ignore_index=True)
id_matching.to_csv('/content/drive/MyDrive/fly_project/id_matching_17.csv', index=False)
id_matching[['role', 'old_id', 'new_id', 'confidence', 'cell_type_v630', 'cell_type_v783', 'out_sign_same', 'status']]

,role,old_id,new_id,confidence,cell_type_v630,cell_type_v783,out_sign_same,status
0,TH_VUM input,720575940616857174,720575940610482883,0.997919,CB3958,CB3958,True,matched
1,right MN9 readout,720575940645521262,720575940618238523,0.999673,CB0701,CB0701,True,matched
2,diatom input,720575940614897458,720575940633968045,0.991266,CB4148,CB4148,True,matched
3,kitty input,720575940610056772,720575940626798736,0.959430,CB0370,CB0370,True,matched
4,marge input,720575940610690372,720575940633941293,0.999185,CB0812,CB0812,True,matched
5,peacock input,720575940631237203,720575940627081534,0.991292,CB0746,CB0746,True,matched


Build the input lists and save them.

In [52]:
replace = dict(zip(r['old_id'].astype('int64'), r['new_id'].astype('int64')))
remaining = ['sink_sync', 'G2N_1', 'clavicle', 'Fdg', 'bract', 'rattle', 'phantom',
             'tentacular', 'usnea', 'peahen', 'diatom', 'kitty', 'marge', 'peacock']
inputs = {}
for n in remaining:
    ids = [int(replace.get(int(i), int(i))) for i in sez[lookup[n.casefold()]]]
    assert all(i in ids_783 for i in ids) and len(set(ids)) == len(ids), n
    inputs[n] = ids
inputs['marge_survivor_only'] = [int(i) for i in sez[lookup['marge']] if int(i) in ids_783]

with open(f"{config['path_res']}/inputs_batch1.json", 'w') as f:
    json.dump(inputs, f, indent=1)
print({k: len(v) for k, v in inputs.items()})

{'sink_sync': 4, 'G2N_1': 2, 'clavicle': 2, 'Fdg': 2, 'bract': 4, 'rattle': 2, 'phantom': 2, 'tentacular': 2, 'usnea': 2, 'peahen': 4, 'diatom': 14, 'kitty': 4, 'marge': 2, 'peacock': 4, 'marge_survivor_only': 1}


Run the other 14 cell types, plus the check that stimulates only marge's surviving v630 neuron. Each experiment is added to the run record when it finishes, so the loop can pick up where it left off after a disconnect.

In [53]:
params['r_poi'] = 50 * Hz
params['n_run'] = 30
with open(f"{config['path_res']}/inputs_batch1.json") as f:
    inputs = json.load(f)

for n, ids in inputs.items():
    with open(mpath) as f:
        manifest = json.load(f)
    if n in manifest['experiments']:
        continue
    base = n.replace('_survivor_only', '')
    started = now()
    run_exp(exp_name=f'{n}_50Hz', neu_exc=ids, params=params, **config)
    entry = {'input_ids': ids, 'input_ids_v630': [int(i) for i in sez[lookup[base.casefold()]]],
             'started': started, 'finished': now()}
    if base in ('diatom', 'kitty', 'marge', 'peacock') and n == base:
        entry['matched_by'] = 'id_matching_17.csv'
    if n == 'marge_survivor_only':
        entry['note'] = 'secondary check: surviving v630 input only'
    manifest['experiments'][n] = entry
    with open(mpath, 'w') as f:
        json.dump(manifest, f, indent=1)

>>> Skipping experiment sink_sync_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/sink_sync_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment G2N_1_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/G2N_1_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment clavicle_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/clavicle_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment Fdg_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/Fdg_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment bract_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/bract_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment rattle_50Hz because /content/drive/MyDrive/fly_project/results_v783/batch1/rattle_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment phantom_50Hz because /content/drive/MyDrive/fly_proj

Compare the two releases for all 17 cell types. The main call is any left MN9 spike in the 30 trials. The saved CSV also has the calls under the stricter >1 Hz rule and under a rule that counts a spike on either side.

In [54]:
import pandas as pd
MN9_RIGHT_783 = 720575940618238523
v630 = pd.read_csv('/content/drive/MyDrive/fly_project/results_v630/v630_50Hz_MN9_summary.csv').set_index('cell_type')

rows = []
for n in list(v630.index) + ['marge_survivor_only']:
    df = pd.read_parquet(f"{config['path_res']}/{n}_50Hz.parquet", columns=['trial', 'flywire_id'])
    assert df['trial'].nunique() == 30, n
    L = df[df['flywire_id'] == MN9_LEFT]
    R = df[df['flywire_id'] == MN9_RIGHT_783]
    b = v630.loc[n.replace('_survivor_only', '')]
    rows.append({'cell_type': n, 'lab_fed': b['observed_fraction'] > 0,
                 'v630_left_spikes': int(b['left_spikes']), 'v630_left_trials': int(b['left_trials_fired']),
                 'v783_left_spikes': len(L), 'v783_left_trials': L['trial'].nunique(),
                 'v630_right_spikes': int(b['right_spikes']), 'v783_right_spikes': len(R)})
t = pd.DataFrame(rows)

t['v630_call'] = t['v630_left_spikes'] > 0
t['v783_call'] = t['v783_left_spikes'] > 0
t['flip'] = t['v630_call'] != t['v783_call']
names = {(True, True): 'stayed correct', (False, False): 'stayed wrong',
         (False, True): 'improved', (True, False): 'worsened'}
t['outcome'] = [names[(a == lab, b == lab)] for a, b, lab in zip(t['v630_call'], t['v783_call'], t['lab_fed'])]
t['v630_call_over_1hz'] = t['v630_left_spikes'] / 30 > 1
t['v783_call_over_1hz'] = t['v783_left_spikes'] / 30 > 1
t['v630_call_either'] = t['v630_left_spikes'] + t['v630_right_spikes'] > 0
t['v783_call_either'] = t['v783_left_spikes'] + t['v783_right_spikes'] > 0
t.to_csv(f"{config['path_res']}/v783_comparison_17.csv", index=False)

main = t[t['cell_type'] != 'marge_survivor_only']
print(main['outcome'].value_counts().to_string())
print("Flips (main rule):", main.loc[main['flip'], 'cell_type'].tolist())
t[['cell_type', 'lab_fed', 'v630_left_spikes', 'v630_left_trials', 'v783_left_spikes',
   'v783_left_trials', 'v630_right_spikes', 'v783_right_spikes', 'flip', 'outcome']]

outcome
stayed correct    11
stayed wrong       6
Flips (main rule): []


,cell_type,lab_fed,v630_left_spikes,v630_left_trials,v783_left_spikes,v783_left_trials,v630_right_spikes,v783_right_spikes,flip,outcome
0,roundup,True,2363,30,2314,30,2468,2391,False,stayed correct
1,diatom,True,914,30,874,30,922,923,False,stayed correct
2,sink_sync,True,800,30,743,30,828,766,False,stayed correct
3,G2N_1,True,275,30,351,30,292,383,False,stayed correct
4,clavicle,True,300,30,314,30,202,201,False,stayed correct
5,Fdg,True,687,30,703,30,628,618,False,stayed correct
6,bract,True,978,30,1067,30,727,770,False,stayed correct
7,rattle,True,77,27,111,30,41,48,False,stayed correct
8,FMIn,True,18,10,8,8,5,4,False,stayed correct
9,TH_VUM,True,0,0,0,0,78,101,False,stayed wrong


Repair the run record. Run all was clicked again after the batch had finished, which rewrote the times in the manifest. This cell takes each finish time from its output file and adds a note saying so.

In [55]:
import os, json, datetime
with open(mpath) as f:
    manifest = json.load(f)
for n, e in manifest['experiments'].items():
    p = f"{config['path_res']}/{n}_50Hz.parquet"
    e['finished'] = datetime.datetime.fromtimestamp(os.path.getmtime(p)).isoformat(timespec='seconds')
    e.pop('started', None)
manifest['note'] = ("Run all was re-run by accident after the batch finished. Simulations were skipped "
                    "because their output files existed. 'finished' times come from the output files' "
                    "modification times; start times were not kept.")
with open(mpath, 'w') as f:
    json.dump(manifest, f, indent=1)
{n: e['finished'] for n, e in manifest['experiments'].items()}

{'roundup': '2026-11-23T21:05:57',
 'FMIn': '2026-11-23T21:10:02',
 'TH_VUM': '2026-11-23T21:42:02',
 'sink_sync': '2026-11-23T22:05:45',
 'G2N_1': '2026-11-23T22:10:13',
 'clavicle': '2026-11-23T22:13:38',
 'Fdg': '2026-11-23T22:16:45',
 'bract': '2026-11-23T22:21:39',
 'rattle': '2026-11-23T22:24:39',
 'phantom': '2026-11-23T22:27:38',
 'tentacular': '2026-11-23T22:31:06',
 'usnea': '2026-11-23T22:35:19',
 'peahen': '2026-11-23T22:38:50',
 'diatom': '2026-11-23T22:47:50',
 'kitty': '2026-11-23T22:52:10',
 'marge': '2026-11-23T22:55:15',
 'peacock': '2026-11-23T22:58:56',
 'marge_survivor_only': '2026-11-23T23:02:51'}